In [0]:
WITH monthly_sales AS (
    SELECT
        d.year,
        d.month,
        d.year_month,

        SUM(f.price) AS monthly_sales,

        COUNT(DISTINCT f.order_id) AS monthly_orders

    FROM `e-commerce`.ecommerce_gold.fact_order_items f

    JOIN `e-commerce`.ecommerce_gold.dim_date d
        ON f.purchase_date_key = d.date_key

    GROUP BY
        d.year,
        d.month,
        d.year_month
),

growth AS (
    SELECT
        *,

        LAG(monthly_sales) OVER (
            ORDER BY year, month
        ) AS previous_month_sales,

        LAG(monthly_orders) OVER (
            ORDER BY year, month
        ) AS previous_month_orders

    FROM monthly_sales
)

SELECT
    year_month,

    ROUND(monthly_sales, 2) AS monthly_sales,

    monthly_orders,

    ROUND(
        100.0 * (monthly_sales - previous_month_sales)
        / NULLIF(previous_month_sales, 0),
        2
    ) AS sales_mom_growth_pct,

    ROUND(
        100.0 * (monthly_orders - previous_month_orders)
        / NULLIF(previous_month_orders, 0),
        2
    ) AS orders_mom_growth_pct

FROM growth

ORDER BY year, month;